# Template Based Diagram Layouts
This notebook is intended to explore the use of template-based diagram layouts 
for visualizing relationships between structures in a dataset. 
The goal is to compare this approach with the current optimization-based layout 
and also to evaluate the use to Networkx for layouts.

## Setup

### Imports

In [1]:
from typing import Dict, List, Union

from pathlib import Path
import re

from pprint import pprint

In [2]:
import xml.etree.ElementTree as ET
from itertools import chain


In [3]:
import pandas as pd
import xlwings as xw
import networkx as nx
from matplotlib import pyplot as plt

In [4]:
from structure_set import StructureSet
from dicom import DicomStructureFile
from structure_id_parser import parse_structure_metadata

INFO:metrics.base:Registered calculator: minimum_margins (ContainmentMarginsCalculator)
INFO:metrics.base:Registered calculator: maximum_margin (MaximumMarginCalculator)
INFO:metrics.base:Registered calculator: minimum_distance (MinimumDistanceCalculator)


### Paths

In [5]:
base_path = Path.cwd()
dicom_path = base_path / "tests"
structure_filter_def = base_path / "src" / "webapp" / "config" / "structure_filter_rules.json"

## Utility Functions

### Load the DICOM file

In [6]:
def load_structures(dicom_path: Path, dcm_file_path: Path,
                    apply_filter=False)->pd.DataFrame:
    '''
    Load structure names from a DICOM file and optionally apply a filter.

    Args:
        dicom_path (Path): The directory containing the DICOM file.
        dcm_file_path (Path): The path to the DICOM file.
        apply_filter (bool): Whether to apply a filter to the structures.
            Default is False.

    Returns:
        pd.DataFrame: A DataFrame containing the structure IDs and their metadata.
    '''
    dicom_file = DicomStructureFile(top_dir=dicom_path, file_path=dcm_file_path)
    #pprint(dicom_file.structure_set_info)

    # get structure IDs
    meta_data = dicom_file.get_structure_filter_metadata()
    if apply_filter:
        filter_report = dicom_file.evaluate_structure_filters(structure_filter_def)
        selection = filter_report.SelectedByDefault & filter_report.DisplayedByDefault
        meta_data = meta_data[selection]
    return meta_data

## Load an example CNS DICOM structure set

### Path to the example DICOM file

In [7]:
dcm_file_name = 'RS.CNS_FSRT_2_GTV.BRAI.dcm'
dcm_file_path = dicom_path / dcm_file_name
structure_data = load_structures(dicom_path, dcm_file_path)
structures_df = parse_structure_metadata(structure_data)

INFO:dicom:Successfully loaded DICOM dataset from RS.CNS_FSRT_2_GTV.BRAI.dcm
INFO:dicom:Extracted 2911 contours from 35 ROIs
INFO:dicom:Found 0 frame-of-reference matches and 0 other matches for structure set RS.CNS_FSRT_2_GTV.BRAI.dcm
INFO:dicom:Calculated resolution from structure 'BODY': 0.040 cm/pixel


In [8]:
structures_df


,Mod,TargetType,TargetNumber,TargetDose,DoseUnits,FractionDelimiter,Fractions,Combined,TargetOrgan,ExpansionSize,DICOM Type,Structure Code,Coding Scheme,Code Meaning
Structure,,,,,,,,,,,,,,
CTV,NaN,CTV,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CTV,CTVp,99VMS_STRUCTCODE,Primary Clinical Target Volume
GTV 1 xxGy,NaN,GTV,1,NaN,NaN,NaN,NaN,NaN,xxGy,NaN,GTV,GTVp,99VMS_STRUCTCODE,Primary Gross Tumor Volume
GTV 2 xxGy,NaN,GTV,2,NaN,NaN,NaN,NaN,NaN,xxGy,NaN,GTV,GTVp,99VMS_STRUCTCODE,Primary Gross Tumor Volume
GTV Total,NaN,GTV,NaN,NaN,NaN,NaN,NaN,Total,NaN,NaN,GTV,GTVp,99VMS_STRUCTCODE,Primary Gross Tumor Volume
PTV 1_24Gyin3,NaN,PTV,1,24,Gy,in,3,NaN,NaN,NaN,PTV,PTVp,99VMS_STRUCTCODE,Primary Planning Target Volume
PTV 2_24Gyin3,NaN,PTV,2,24,Gy,in,3,NaN,NaN,NaN,PTV,PTVp,99VMS_STRUCTCODE,Primary Planning Target Volume
PTV Total,NaN,PTV,NaN,NaN,NaN,NaN,NaN,Total,NaN,NaN,PTV,PTV_High,99VMS_STRUCTCODE,Planning Target Volume High Risk
PTV+15,NaN,PTV,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15,TREATED_VOLUME,Treated Volume,99VMS_STRUCTCODE,Treated Volume
shell_PTV 2_24Gyin3,shell,PTV,2,24,Gy,in,3,NaN,NaN,NaN,CONTROL,,,
